# Appeler une bibliothèque C avec `ctypes`

Ce notebook appelle depuis Python les fonctions d'une petite bibliothèque C, écrite pour vous : vous n'écrivez que du Python.

1. Charger la bibliothèque et déclarer les types
2. Passer un tableau NumPy à une fonction C
3. Récupérer un tableau rempli par le C
4. Plus rapide que Python ?
5. Pour aller plus loin : le coût d'un appel

Les étapes s'enchaînent : chaque exercice se sert de ce que les solutions des précédents définissent. Chaque exercice a une cellule pour votre code, puis des conseils supplémentaires et sa solution, repliés : ouvrez-les si besoin.

Le notebook compile du C avec `gcc`, installé dans Colab comme dans le Codespace.

## La bibliothèque C

La cellule suivante écrit le fichier `signal.c`, qui définit trois fonctions :

- `interpole(a, b, t)` renvoie le point situé à la fraction `t` du chemin de `a` à `b` ;
- `moyenne(x, n)` renvoie la moyenne des `n` valeurs du tableau `x` ;
- `lisse(x, y, n, k)` écrit dans `y` les moyennes glissantes de `x` sur des fenêtres de `k` valeurs consécutives, soit `n - k + 1` moyennes, et renvoie `0`, ou `-1` si `k` n'est pas entre `1` et `n`. D'une fenêtre à la suivante, elle ajoute la valeur qui entre et retire celle qui sort.

In [ ]:
%%writefile signal.c
#include <math.h>

double interpole(double a, double b, double t) {
  return a + (b - a) * t;
}

double moyenne(const double *x, int n) {
  if (n <= 0) {
    return NAN;
  }
  double somme = 0.0;
  for (int i = 0; i < n; i++) {
    somme += x[i];
  }
  return somme / n;
}

int lisse(const double *x, double *y, int n, int k) {
  if (k <= 0 || k > n) {
    return -1;
  }
  double somme = 0.0;
  for (int i = 0; i < k; i++) {
    somme += x[i];
  }
  y[0] = somme / k;
  for (int i = k; i < n; i++) {
    somme += x[i] - x[i - k];
    y[i - k + 1] = somme / k;
  }
  return 0;
}

La suivante la compile en bibliothèque partagée, `libsignal.so` : `-shared` produit une bibliothèque plutôt qu'un exécutable, `-fPIC` du code qui fonctionne à l'adresse où elle sera chargée.

In [ ]:
!gcc -O2 -Wall -fPIC -shared signal.c -o libsignal.so

## 1. Charger la bibliothèque et déclarer les types

La cellule suivante charge `libsignal.so` avec `ctypes.CDLL` et appelle `interpole` sans rien déclarer : `ctypes` ne sait pas convertir un `float` Python, et l'appel échoue.

In [ ]:
import ctypes

import numpy as np

lib = ctypes.CDLL("./libsignal.so")

try:
  lib.interpole(2.0, 4.0, 0.25)
except ctypes.ArgumentError as erreur:
  print("Erreur :", erreur)

Déclarez les types des arguments (`argtypes`) et du résultat (`restype`) de `lib.interpole`, d'après sa signature C, puis affichez `lib.interpole(2.0, 4.0, 0.25)` et `lib.interpole(10, 20, 0.5)`. Vous devez obtenir `2.5` et `15.0` : une fois les types déclarés, `ctypes` convertit aussi les entiers.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`a`, `b` et `t` sont des `double` C, soit `ctypes.c_double`, comme le résultat. `argtypes` est une liste, un type par argument, dans l'ordre ; `restype` est un seul type.

Sans `restype`, `ctypes` lit le résultat comme un `int` C : essayez, l'appel renvoie un entier sans rapport avec `2.5`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
lib.interpole.argtypes = [...]
lib.interpole.restype = ...
print(lib.interpole(2.0, 4.0, 0.25))
print(lib.interpole(10, 20, 0.5))
```

### Solution

In [ ]:
lib.interpole.argtypes = [ctypes.c_double, ctypes.c_double, ctypes.c_double]
lib.interpole.restype = ctypes.c_double
print(lib.interpole(2.0, 4.0, 0.25))
print(lib.interpole(10, 20, 0.5))

## 2. Passer un tableau NumPy à une fonction C

`moyenne` reçoit un pointeur vers le premier élément d'un tableau de `double` et sa longueur. Déclarez ses types, puis écrivez une fonction Python `moyenne(valeurs)` qui accepte une liste ou un tableau NumPy, le convertit en tableau NumPy contigu de `float64` et appelle `lib.moyenne` avec un pointeur vers ses données. Affichez `moyenne([1, 2, 3, 4])`, qui vaut `2.5`, puis `moyenne(mesures)` et `np.mean(mesures)`, qui doivent coïncider. Que renvoie `moyenne([])` ?

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.ascontiguousarray.html
- https://numpy.org/doc/stable/reference/generated/numpy.ndarray.ctypes.html

In [ ]:
rng = np.random.default_rng(0)
t = np.linspace(0, 1, 1_000)
mesures = np.sin(2 * np.pi * 3 * t) + rng.normal(0, 0.3, t.size)

# Votre code ici

### Conseils supplémentaires

- `ctypes.POINTER(ctypes.c_double)` est le type « pointeur vers un `double` », `ctypes.c_int` celui de `n`, et le résultat est un `ctypes.c_double`.
- `np.ascontiguousarray(valeurs, dtype=np.float64)` renvoie un tableau dont les valeurs sont des `double` C rangés côte à côte en mémoire, ce qu'attend le C (sans copie si c'est déjà le cas).
- `tableau.ctypes.data_as(ctypes.POINTER(ctypes.c_double))` donne le pointeur vers ses données.

`moyenne([])` renvoie `nan` : le C renvoie `NAN` quand `n` vaut 0, comme `np.mean`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
lib.moyenne.argtypes = [..., ...]
lib.moyenne.restype = ...


def moyenne(valeurs) -> float:
  tableau = np.ascontiguousarray(...)
  pointeur = tableau.ctypes.data_as(...)
  return lib.moyenne(pointeur, ...)


print(moyenne([1, 2, 3, 4]))
print(moyenne(mesures), np.mean(mesures))
print(moyenne([]))
```

### Solution

In [ ]:
lib.moyenne.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.c_int]
lib.moyenne.restype = ctypes.c_double


def moyenne(valeurs) -> float:
  tableau = np.ascontiguousarray(valeurs, dtype=np.float64)
  pointeur = tableau.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
  return lib.moyenne(pointeur, len(tableau))


print(moyenne([1, 2, 3, 4]))
print(moyenne(mesures), np.mean(mesures))
print(moyenne([]))

## 3. Récupérer un tableau rempli par le C

`lisse` écrit son résultat dans un tableau `y` que l'appelant lui fournit. Cette fois, déclarez ses deux tableaux avec `np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS")` : `ctypes` accepte alors un tableau NumPy tel quel, et vérifie son type. Écrivez `lisse(valeurs, k)`, qui renvoie un nouveau tableau NumPy des `len(valeurs) - k + 1` moyennes glissantes et lève une `ValueError` quand le C renvoie `-1`. Vérifiez avec `np.allclose` que `lisse(mesures, 5)` donne le même résultat que `np.convolve(mesures, np.ones(5) / 5, mode="valid")`, puis que `lisse(mesures, 0)` lève bien une `ValueError`.

**Documentation utile**

- https://numpy.org/doc/stable/reference/routines.ctypeslib.html#numpy.ctypeslib.ndpointer
- https://numpy.org/doc/stable/reference/generated/numpy.empty_like.html
- https://numpy.org/doc/stable/reference/generated/numpy.allclose.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les types : `[tableau_double, tableau_double, ctypes.c_int, ctypes.c_int]`, où `tableau_double` est le type que renvoie `np.ctypeslib.ndpointer`, et `ctypes.c_int` pour le résultat.
- `lisse` convertit toujours son argument avec `np.ascontiguousarray` : le type `ndpointer` refuse une liste (`ctypes.ArgumentError`) et un tableau d'entiers.
- `np.empty_like(x)` alloue `y`, de la longueur de `x`, sans l'initialiser : le C en remplit les `len(x) - k + 1` premières cases, que `lisse` renvoie.

Exemple : `lisse([1, 2, 3, 4, 5], 2)` renvoie `array([1.5, 2.5, 3.5, 4.5])`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
tableau_double = np.ctypeslib.ndpointer(dtype=..., flags="C_CONTIGUOUS")
lib.lisse.argtypes = [...]
lib.lisse.restype = ...


def lisse(valeurs, k: int) -> np.ndarray:
  x = np.ascontiguousarray(valeurs, dtype=np.float64)
  y = np.empty_like(x)
  if ... != 0:
    msg = f"fenêtre invalide : {k}"
    raise ValueError(msg)
  return y[...]


print(lisse([1, 2, 3, 4, 5], 2))
reference = np.convolve(mesures, np.ones(5) / 5, mode="valid")
print(np.allclose(..., reference))
try:
  lisse(mesures, 0)
except ValueError as erreur:
  print("Erreur :", erreur)
```

### Solution

In [ ]:
tableau_double = np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS")
lib.lisse.argtypes = [tableau_double, tableau_double, ctypes.c_int, ctypes.c_int]
lib.lisse.restype = ctypes.c_int


def lisse(valeurs, k: int) -> np.ndarray:
  x = np.ascontiguousarray(valeurs, dtype=np.float64)
  y = np.empty_like(x)
  if lib.lisse(x, y, len(x), k) != 0:
    msg = f"fenêtre invalide : {k}"
    raise ValueError(msg)
  return y[: len(x) - k + 1]


print(lisse([1, 2, 3, 4, 5], 2))
reference = np.convolve(mesures, np.ones(5) / 5, mode="valid")
print(np.allclose(lisse(mesures, 5), reference))
try:
  lisse(mesures, 0)
except ValueError as erreur:
  print("Erreur :", erreur)

## 4. Plus rapide que Python ?

La cellule suivante définit `lisse_python`, le même calcul en Python pur, et un signal d'un million de valeurs, `signal_long`, et sa copie en liste Python, `liste_longue`. Mesurez avec `%timeit` le temps de `lisse_python` sur `liste_longue`, puis de votre `lisse` et de `np.convolve` (comme au 3) sur `signal_long`, avec des fenêtres de 50 valeurs. Lequel est le plus rapide ? Pourquoi `np.convolve`, écrit en C lui aussi, est-il plus lent que `lisse` ?

**Documentation utile**

- https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit

In [ ]:
def lisse_python(valeurs: list[float], k: int) -> list[float]:
  somme = sum(valeurs[:k])
  resultat = [somme / k]
  for i in range(k, len(valeurs)):
    somme += valeurs[i] - valeurs[i - k]
    resultat.append(somme / k)
  return resultat


signal_long = rng.normal(size=1_000_000)
liste_longue = signal_long.tolist()

# Votre code ici

### Conseils supplémentaires

`%timeit` suivi d'une expression l'exécute plusieurs fois et affiche son temps moyen. `lisse_python` reçoit une liste : une boucle Python sur un tableau NumPy est encore plus lente, chaque élément lu devenant un objet Python.

Pour `np.convolve`, comptez les opérations : combien de multiplications pour une moyenne, et combien d'opérations fait `lisse` ?

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
%timeit lisse_python(liste_longue, 50)
%timeit lisse(...)
%timeit np.convolve(...)
```

### Solution

In [ ]:
%timeit lisse_python(liste_longue, 50)
%timeit lisse(signal_long, 50)
%timeit np.convolve(signal_long, np.ones(50) / 50, mode="valid")
# Le C est des dizaines de fois plus rapide que la boucle Python. np.convolve,
# compilé lui aussi, fait 50 multiplications par moyenne, quand lisse ajoute
# une valeur et en retire une : c'est l'algorithme qui fait la différence.

## 5. Pour aller plus loin : le coût d'un appel

Chaque appel d'une fonction C par `ctypes` convertit ses arguments et son résultat. Mesurez avec `%timeit` le calcul de la liste des `lib.interpole(0.0, 1.0, t)` pour chaque `t` de `positions` (100 000 valeurs), puis celui de la même liste en Python, `0.0 + (1.0 - 0.0) * t`. Qu'en concluez-vous sur la façon de partager le travail entre Python et le C ?

In [ ]:
positions = np.linspace(0, 1, 100_000).tolist()

# Votre code ici

### Conseils supplémentaires

Une liste en compréhension, `[... for t in positions]`, derrière `%timeit`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
%timeit [lib.interpole(0.0, 1.0, t) for t in positions]
%timeit [... for t in positions]
```

### Solution

In [ ]:
%timeit [lib.interpole(0.0, 1.0, t) for t in positions]
%timeit [0.0 + (1.0 - 0.0) * t for t in positions]
# Appeler le C est plus de dix fois plus lent que de calculer en Python :
# convertir les arguments et le résultat coûte bien plus que le calcul. On
# confie donc au C tout un tableau en un appel, comme lisse, et non une valeur
# à la fois.